# Contextual Chunk Headers for RAG

## Overview

**Contextual Chunk Headers (CCH)** is a retrieval-augmented generation technique that prepends
hierarchical document context — such as document title, chapter heading, and section name — to
each text chunk *before* it is embedded. This simple yet powerful pre-processing step restores
the context that is lost when a long document is split into small pieces, dramatically improving
both retrieval precision and downstream generation quality.

## Why This Matters

When a document is chunked for vector search, each chunk becomes an island:

> *"This increased by 15% year-over-year, driven primarily by expansion in the APAC region."*

**What** increased? Revenue? Headcount? Carbon emissions? Without the surrounding document
structure the embedding model has no way to disambiguate, and neither does the reader.

CCH solves this by physically attaching the missing hierarchy to every chunk so the embedding
captures the full semantic meaning.

## What You Will Learn

1. **The Lost Context Problem** — concrete examples of chunks that lose meaning after splitting.
2. **Header Extraction** — parsing document structure into a hierarchy (Document → Chapter → Section → Subsection).
3. **Implementation from Scratch** — building a chunk-to-header mapping and prepending headers.
4. **Embedding Comparison** — measuring cosine similarity with and without headers.
5. **Graded Headers** — comparing different levels of context (full hierarchy vs. section-only vs. title-only).
6. **Complete RAG Pipeline** — header-enriched chunks → FAISS → retrieve → generate.
7. **Synthesis** — when CCH helps most, overhead trade-offs, and production patterns.

## Visual Intuition

```
┌──────────────────────────────────────────────────────────┐
│  Original Document                                       │
│  ┌─ Chapter 3: Revenue Analysis                         │
│  │  ┌─ Section 3.2: Regional Breakdown                  │
│  │  │   "This increased by 15% year-over-year,          │
│  │  │    driven primarily by APAC expansion."            │
│  │  └───────────────────────────────────────             │
│  └──────────────────────────────────────────             │
└──────────────────────────────────────────────────────────┘

        │ Chunking WITHOUT headers          │ Chunking WITH headers
        ▼                                   ▼
┌─────────────────────┐   ┌──────────────────────────────────────────┐
│ "This increased by  │   │ Document: Acme Corp 2024 Annual Report   │
│  15% year-over-year,│   │ Chapter: Revenue Analysis                │
│  driven primarily   │   │ Section: Regional Breakdown              │
│  by APAC expansion."│   │                                          │
│                     │   │ "This increased by 15% year-over-year,   │
│ ❌ WHAT increased?  │   │  driven primarily by APAC expansion."    │
└─────────────────────┘   │                                          │
                          │ ✅ Revenue increased — clear!            │
                          └──────────────────────────────────────────┘
```

The header acts as a **semantic anchor**: the embedding now encodes *"revenue in the APAC region
increased by 15%"* rather than a vague *"something increased by 15%."*

## Setup

Install dependencies, mount Google Drive for model caching, and load the LLM + embedding model.

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy

import torch, gc
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

MODEL_NAME = "Qwen/Qwen3-14B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample, top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print("✅ LLM loaded:", MODEL_NAME)

In [ ]:
# --- Load Embedding Model ---
from sentence_transformers import SentenceTransformer
import numpy as np

EMBED_MODEL_NAME = "BAAI/bge-base-en-v1.5"
embed_model = SentenceTransformer(EMBED_MODEL_NAME, cache_folder=CACHE_DIR)

def embed_texts(texts):
    """Embed a list of strings and return a numpy array of shape (n, dim)."""
    return embed_model.encode(texts, normalize_embeddings=True, show_progress_bar=False)

sample = embed_texts(["Hello world"])
print(f"✅ Embedding model loaded: {EMBED_MODEL_NAME}")
print(f"   Embedding dimension: {sample.shape[1]}")

---

# Part 1 — The Lost Context Problem

Imagine you have a 50-page annual report. After chunking it into ~500-character pieces you get
fragments like:

| Chunk | Text |
|-------|------|
| 42 | *"This was primarily driven by a 12% increase in unit volume."* |
| 87 | *"The committee recommended a restructuring of the division."* |
| 115 | *"Margins contracted by 200 basis points compared to the prior year."* |

**Every one of these chunks is ambiguous in isolation.** What had a 12% increase? Which
committee? Which division? Whose margins contracted?

The embedding model encodes *exactly what it sees*. If the chunk says "this increased" the
embedding captures "something increased" — which matches far too many queries and misses
the specific ones it should match.

Let's demonstrate this concretely.

In [ ]:
# --- Demonstrate the Lost Context Problem ---
from numpy.linalg import norm

def cosine_sim(a, b):
    """Cosine similarity between two vectors."""
    return float(np.dot(a, b) / (norm(a) * norm(b)))

# An ambiguous chunk (no context)
ambiguous_chunk = (
    "This increased by 15% year-over-year, driven primarily by "
    "expansion in the APAC region and new enterprise contracts."
)

# A correctly-contextualized version
contextualized_chunk = (
    "Document: Acme Corp 2024 Annual Report\n"
    "Chapter: Financial Performance\n"
    "Section: Revenue Analysis — Regional Breakdown\n\n"
    "This increased by 15% year-over-year, driven primarily by "
    "expansion in the APAC region and new enterprise contracts."
)

queries = [
    "Acme Corp revenue growth in Asia Pacific",
    "headcount increase in APAC region",
    "carbon emissions year-over-year change",
]

emb_amb   = embed_texts([ambiguous_chunk])[0]
emb_ctx   = embed_texts([contextualized_chunk])[0]
emb_queries = embed_texts(queries)

print("Query".ljust(48), "w/o Header", " w/ Header", " Δ")
print("-" * 82)
for q, eq in zip(queries, emb_queries):
    sim_amb = cosine_sim(emb_amb, eq)
    sim_ctx = cosine_sim(emb_ctx, eq)
    delta = sim_ctx - sim_amb
    print(f"{q:<48} {sim_amb:>8.4f}   {sim_ctx:>8.4f}   {delta:>+.4f}")

print()
print("Notice: the correct query ('revenue growth') gets a BIGGER boost from the header,")
print("while unrelated queries ('headcount', 'emissions') get a smaller or negative boost.")
print("Headers help the embedding *disambiguate* what the chunk is actually about.")

---

# Part 2 — What Are Contextual Chunk Headers?

**Contextual Chunk Headers (CCH)** is the practice of prepending document-level and
section-level metadata to each chunk *before* embedding. The header typically includes:

| Level | Example |
|-------|---------|
| **Document Title** | `Acme Corp 2024 Annual Report` |
| **Chapter** | `Chapter 3: Financial Performance` |
| **Section** | `3.2 Revenue Analysis` |
| **Subsection** | `3.2.1 Regional Breakdown` |

The concatenation `header + "\n\n" + chunk_text` is what gets embedded (and later
presented to the LLM during generation).

### Why does this work?

Modern embedding models encode the **full input context**. When the header says
"Revenue Analysis" the embedding shifts toward the revenue-related region of the
vector space — exactly where queries about revenue live. This is analogous to how
a human reader uses chapter headings to understand what a paragraph is about.

### Key design choices

1. **Separator**: Use a clear delimiter (e.g., double newline) between header and body.
2. **Hierarchy depth**: More levels = more precision, but also more tokens per chunk.
3. **Embedding + generation**: Use the *same* enriched text for both embedding and LLM context.

---

# Part 3 — Header Extraction from Structured Documents

To demonstrate CCH properly, we need a document with clear hierarchical structure.
We'll create a synthetic multi-chapter technical report inline. This gives us full
control over the hierarchy and lets us verify our parsing is correct.

In [ ]:
# --- Create a synthetic structured document ---
DOCUMENT = """
# Acme Corp 2024 Annual Report

## Chapter 1: Company Overview

### 1.1 Mission and Vision

Acme Corp is a global technology company headquartered in San Francisco. Our mission is to
empower businesses through intelligent automation. Founded in 2010, we have grown from a
small startup to a Fortune 500 company with operations in 45 countries. Our vision is to
make AI accessible to every organization regardless of size or technical sophistication.

### 1.2 Key Milestones

In 2024, Acme Corp achieved several significant milestones. We crossed the $10 billion
annual revenue mark for the first time. Our customer base expanded to over 50,000
enterprise clients. We launched the Acme AI Platform v3.0, which saw adoption rates
3x higher than the previous version in its first quarter.

## Chapter 2: Financial Performance

### 2.1 Revenue Analysis

Total revenue for fiscal year 2024 was $10.3 billion, representing a 23% increase
year-over-year. Subscription revenue grew 31% to $7.8 billion, now comprising 76% of
total revenue. Professional services revenue declined 5% to $2.5 billion as customers
shifted toward self-service deployment.

### 2.2 Regional Breakdown

North America remained our largest market at $5.2 billion (50% of revenue), growing 18%.
Europe contributed $2.8 billion (27%), growing 25%. The APAC region was the fastest-growing
at $1.8 billion (17%), up 42% year-over-year driven by expansion in Japan, South Korea,
and Australia. Rest of World contributed $0.5 billion (5%), growing 15%.

### 2.3 Profitability

Operating income was $2.1 billion with an operating margin of 20.4%, up from 18.1% in
the prior year. Net income reached $1.7 billion. Margins expanded due to economies of
scale in cloud infrastructure and a shift toward higher-margin subscription products.
The company generated $2.8 billion in free cash flow.

## Chapter 3: Product and Technology

### 3.1 AI Platform v3.0

The Acme AI Platform v3.0 was released in March 2024. Key features include multi-modal
reasoning, a new retrieval-augmented generation engine, and support for on-premises
deployment. The platform processes over 500 million API calls per day. Customer
satisfaction scores improved from 4.1 to 4.6 out of 5.0 after the upgrade.

### 3.2 Research and Development

R&D expenditure was $1.8 billion (17% of revenue), up from $1.5 billion in the prior
year. The research team published 47 papers at top-tier conferences. Key research areas
include efficient inference, multi-agent systems, and safety alignment. We filed 312
new patents in 2024, bringing the total portfolio to 2,100 patents.

### 3.3 Security and Compliance

The platform achieved SOC 2 Type II, ISO 27001, and FedRAMP High certifications. Zero
critical security incidents were reported in 2024. The bug bounty program received 1,200
submissions and paid out $2.3 million in rewards. Data residency options expanded to cover
18 geographic regions.

## Chapter 4: People and Culture

### 4.1 Workforce Growth

Total headcount reached 28,500 employees, a 22% increase from the prior year. Engineering
headcount grew 30% to 14,200. We hired across 35 countries. The voluntary attrition rate
decreased from 14% to 11%, well below the industry average of 18%.

### 4.2 Diversity and Inclusion

Women now represent 38% of the global workforce, up from 34%. Underrepresented minorities
in US technical roles increased from 18% to 22%. The company invested $45 million in
diversity programs including mentorship, scholarship funds, and partnerships with HBCUs.

### 4.3 Employee Benefits

The company introduced unlimited mental health counseling, a 16-week parental leave
policy, and a $5,000 annual learning stipend. Employee Net Promoter Score improved from
+32 to +48. Remote and hybrid work options are available to 85% of employees.

## Chapter 5: Risk Factors

### 5.1 Market Competition

The enterprise AI market is intensely competitive. Major cloud providers (AWS, Azure, GCP)
offer competing products, often bundled with existing cloud services at discounted rates.
Open-source alternatives continue to improve and may reduce demand for commercial platforms.
New entrants funded by venture capital add further competitive pressure.

### 5.2 Regulatory Environment

AI regulation is evolving rapidly across jurisdictions. The EU AI Act imposes new
compliance requirements for high-risk AI systems. Potential regulation in the United States
and Asia may impose additional requirements. Non-compliance could result in fines of up to
6% of global revenue and reputational damage.

### 5.3 Cybersecurity Threats

Despite strong security posture, the company faces constant cybersecurity threats. State-
sponsored actors and criminal organizations target enterprise software companies. A
significant breach could result in loss of customer trust, regulatory penalties, and
material financial impact. The company invests $180 million annually in cybersecurity.
""".strip()

print(f"Document length: {len(DOCUMENT):,} characters")
print(f"Preview (first 200 chars):")
print(DOCUMENT[:200] + "...")

## Parsing the Document Hierarchy

We need to extract the heading structure so we can map every chunk back to its place
in the hierarchy. Our document uses Markdown headings:

- `# ...` → Document title (level 1)
- `## ...` → Chapter (level 2)
- `### ...` → Section (level 3)

We'll build a parser that walks line-by-line and tracks the current heading stack.

In [ ]:
import re

def parse_document_structure(text):
    """
    Parse a Markdown-structured document into sections with their heading hierarchy.

    Returns a list of dicts:
      { 'hierarchy': ['Doc Title', 'Chapter', 'Section'],
        'text': 'paragraph text...',
        'level': 3 }
    """
    lines = text.split("\n")
    sections = []
    current_hierarchy = {}  # level -> heading text
    current_text_lines = []
    current_level = 0

    def flush():
        nonlocal current_text_lines
        body = "\n".join(current_text_lines).strip()
        if body:
            hierarchy = []
            for lvl in sorted(current_hierarchy.keys()):
                if lvl <= current_level:
                    hierarchy.append(current_hierarchy[lvl])
            sections.append({
                "hierarchy": hierarchy[:],
                "text": body,
                "level": current_level,
            })
        current_text_lines = []

    for line in lines:
        heading_match = re.match(r'^(#{1,4})\s+(.*)', line)
        if heading_match:
            flush()
            hashes = heading_match.group(1)
            level = len(hashes)
            title = heading_match.group(2).strip()
            current_hierarchy[level] = title
            # Remove deeper headings (they're stale now)
            for deeper in list(current_hierarchy.keys()):
                if deeper > level:
                    del current_hierarchy[deeper]
            current_level = level
        else:
            current_text_lines.append(line)

    flush()  # last section
    return sections

sections = parse_document_structure(DOCUMENT)
print(f"Parsed {len(sections)} sections\n")
for sec in sections:
    indent = "  " * (sec['level'] - 1)
    preview = sec['text'][:60].replace("\n", " ")
    print(f"{indent}[L{sec['level']}] {' > '.join(sec['hierarchy'])}")
    print(f"{indent}     "{preview}..."")
    print()

## Chunking with Section Awareness

Now we chunk each section's text. Crucially, we preserve the mapping from each chunk
back to its section's heading hierarchy. We use simple character-based splitting
(no LangChain needed — just Python).

In [ ]:
def chunk_text(text, chunk_size=400, overlap=50):
    """Split text into overlapping chunks by character count."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

def chunk_document_sections(sections, chunk_size=400, overlap=50):
    """
    Chunk each section and build a list of (chunk_text, hierarchy) tuples.
    """
    results = []
    for sec in sections:
        sec_chunks = chunk_text(sec["text"], chunk_size, overlap)
        for c in sec_chunks:
            results.append({
                "text": c,
                "hierarchy": sec["hierarchy"],
            })
    return results

chunked = chunk_document_sections(sections, chunk_size=400, overlap=50)

print(f"Total chunks: {len(chunked)}\n")
for i, c in enumerate(chunked[:5]):
    print(f"Chunk {i}: hierarchy = {c['hierarchy']}")
    print(f"  text = \"{c['text'][:80]}...\"")
    print()

---

# Part 4 — Prepending Contextual Headers

Now the core CCH step: we construct a header string from each chunk's hierarchy and
prepend it to the chunk text. The result is the "enriched chunk" that will be embedded.

### Header format

We use a clean, labeled format:

```
Document: Acme Corp 2024 Annual Report
Chapter: Financial Performance
Section: Revenue Analysis

<chunk text>
```

This format is both human-readable and gives the embedding model explicit semantic cues.

In [ ]:
LEVEL_LABELS = {1: "Document", 2: "Chapter", 3: "Section", 4: "Subsection"}

def build_header(hierarchy):
    """Build a contextual header string from a heading hierarchy list."""
    parts = []
    for i, heading in enumerate(hierarchy):
        level = i + 1
        label = LEVEL_LABELS.get(level, f"Level {level}")
        parts.append(f"{label}: {heading}")
    return "\n".join(parts)

def enrich_chunks(chunked_sections):
    """Prepend contextual headers to each chunk."""
    enriched = []
    for item in chunked_sections:
        header = build_header(item["hierarchy"])
        enriched_text = f"{header}\n\n{item['text']}"
        enriched.append({
            "original": item["text"],
            "header": header,
            "enriched": enriched_text,
            "hierarchy": item["hierarchy"],
        })
    return enriched

enriched_chunks = enrich_chunks(chunked)

# Show example
ex = enriched_chunks[5]
print("=" * 70)
print("ORIGINAL CHUNK (no context):")
print("=" * 70)
print(ex["original"])
print()
print("=" * 70)
print("ENRICHED CHUNK (with contextual header):")
print("=" * 70)
print(ex["enriched"])

## Embedding Comparison: With vs. Without Headers

Let's embed all chunks both ways and measure how headers shift the embeddings relative
to targeted queries. We expect:

1. **Relevant queries** → higher similarity with headers (disambiguation)
2. **Irrelevant queries** → lower or unchanged similarity (less false-positive matching)

In [ ]:
# Embed all chunks both ways
original_texts = [c["original"] for c in enriched_chunks]
enriched_texts = [c["enriched"] for c in enriched_chunks]

emb_original = embed_texts(original_texts)
emb_enriched = embed_texts(enriched_texts)

print(f"Embedded {len(emb_original)} chunks in both modes.")
print(f"Embedding shape: {emb_original.shape}")

In [ ]:
# --- Compare retrieval scores for specific queries ---

test_queries = [
    "Acme Corp revenue growth by region",
    "APAC region revenue performance",
    "employee headcount and attrition rate",
    "AI platform customer satisfaction",
    "cybersecurity investment and threats",
    "operating margin and profitability",
]

emb_queries = embed_texts(test_queries)

print(f"{'Query':<45} {'Best (orig)':>11} {'Best (CCH)':>11} {'Δ':>7}  Best Chunk Section (CCH)")
print("-" * 130)

for q, eq in zip(test_queries, emb_queries):
    sims_orig = emb_original @ eq
    sims_enr  = emb_enriched @ eq

    best_orig_idx = int(np.argmax(sims_orig))
    best_enr_idx  = int(np.argmax(sims_enr))

    best_orig = sims_orig[best_orig_idx]
    best_enr  = sims_enr[best_enr_idx]
    delta = best_enr - best_orig

    section = " > ".join(enriched_chunks[best_enr_idx]["hierarchy"])
    print(f"{q:<45} {best_orig:>11.4f} {best_enr:>11.4f} {delta:>+7.4f}  {section}")

print()
print("Headers consistently improve best-match similarity and help retrieve the correct section.")

### Deep Dive: How Headers Change Similarity Distributions

Let's pick one query and examine how similarity scores shift across *all* chunks.
This reveals that headers don't just boost the best match — they suppress irrelevant ones.

In [ ]:
query = "APAC region revenue performance"
eq = embed_texts([query])[0]

sims_orig = emb_original @ eq
sims_enr  = emb_enriched @ eq

print(f"Query: '{query}'\n")
print(f"{'Metric':<30} {'Original':>10} {'With CCH':>10}")
print("-" * 52)
print(f"{'Max similarity':<30} {sims_orig.max():>10.4f} {sims_enr.max():>10.4f}")
print(f"{'Mean similarity':<30} {sims_orig.mean():>10.4f} {sims_enr.mean():>10.4f}")
print(f"{'Std deviation':<30} {sims_orig.std():>10.4f} {sims_enr.std():>10.4f}")
print(f"{'Max - Mean (signal gap)':<30} {(sims_orig.max()-sims_orig.mean()):>10.4f} {(sims_enr.max()-sims_enr.mean()):>10.4f}")
print()
print("The 'signal gap' (max - mean) grows with CCH, meaning the correct chunk stands out")
print("more clearly from the background — exactly what we want for precise retrieval.")

In [ ]:
# --- Show top-5 retrieved chunks side by side ---

top_k = 5
query = "Acme Corp revenue growth by region"
eq = embed_texts([query])[0]

sims_orig = emb_original @ eq
sims_enr  = emb_enriched @ eq

top_orig = np.argsort(sims_orig)[::-1][:top_k]
top_enr  = np.argsort(sims_enr)[::-1][:top_k]

print(f"Query: '{query}'\n")
print("=" * 70)
print("TOP-5 WITHOUT HEADERS:")
print("=" * 70)
for rank, idx in enumerate(top_orig):
    sec = " > ".join(enriched_chunks[idx]["hierarchy"])
    print(f"  #{rank+1} [sim={sims_orig[idx]:.4f}] {sec}")
    print(f"       \"{enriched_chunks[idx]['original'][:80]}...\"")

print()
print("=" * 70)
print("TOP-5 WITH CONTEXTUAL HEADERS:")
print("=" * 70)
for rank, idx in enumerate(top_enr):
    sec = " > ".join(enriched_chunks[idx]["hierarchy"])
    print(f"  #{rank+1} [sim={sims_enr[idx]:.4f}] {sec}")
    print(f"       \"{enriched_chunks[idx]['original'][:80]}...\"")

print()
print("With headers, the top results are tightly focused on the Revenue/Regional section.")

---

# Part 5 — Graded Headers: How Much Context Is Enough?

Not all applications need the full hierarchy. We'll compare three levels of header richness:

| Level | Header Content | Example |
|-------|---------------|---------|
| **None** | No header | *(raw chunk)* |
| **Title-only** | Just the document title | `Document: Acme Corp 2024 Annual Report` |
| **Full hierarchy** | Document + Chapter + Section | All three levels |

**Hypothesis**: Even the document title alone provides a significant boost, but the full
hierarchy maximizes precision for queries about specific sections.

In [ ]:
def build_graded_header(hierarchy, level="full"):
    """Build headers at different granularity levels."""
    if level == "none":
        return ""
    elif level == "title":
        return f"Document: {hierarchy[0]}" if hierarchy else ""
    elif level == "full":
        return build_header(hierarchy)
    else:
        raise ValueError(f"Unknown level: {level}")

def make_graded_texts(chunked, level):
    texts = []
    for c in chunked:
        header = build_graded_header(c["hierarchy"], level)
        if header:
            texts.append(f"{header}\n\n{c['text']}")
        else:
            texts.append(c["text"])
    return texts

# Embed at each level
texts_none  = make_graded_texts(enriched_chunks, "none")
texts_title = make_graded_texts(enriched_chunks, "title")
texts_full  = make_graded_texts(enriched_chunks, "full")

emb_none  = embed_texts(texts_none)
emb_title = embed_texts(texts_title)
emb_full  = embed_texts(texts_full)

print("Embedded all chunks at 3 header levels: none, title-only, full hierarchy.")

In [ ]:
# --- Compare graded headers across multiple queries ---

graded_queries = [
    "What was Acme Corp total revenue in 2024?",
    "APAC region revenue growth rate",
    "How many patents did Acme Corp file?",
    "Employee diversity statistics",
    "EU AI Act compliance risks",
]

emb_gq = embed_texts(graded_queries)

print(f"{'Query':<45} {'None':>7} {'Title':>7} {'Full':>7}  Best Section (Full)")
print("-" * 120)

for q, eq in zip(graded_queries, emb_gq):
    s_none  = (emb_none  @ eq).max()
    s_title = (emb_title @ eq).max()
    s_full  = (emb_full  @ eq).max()
    best_idx = int(np.argmax(emb_full @ eq))
    sec = " > ".join(enriched_chunks[best_idx]["hierarchy"])
    print(f"{q:<45} {s_none:>7.4f} {s_title:>7.4f} {s_full:>7.4f}  {sec}")

print()
print("Even title-only headers improve over bare chunks.")
print("Full hierarchy provides the strongest signal, especially for section-specific queries.")

In [ ]:
# --- Average improvement across all queries ---

all_queries_text = graded_queries + test_queries
emb_all_q = embed_texts(all_queries_text)

avg_none = np.mean([(emb_none @ eq).max() for eq in emb_all_q])
avg_title = np.mean([(emb_title @ eq).max() for eq in emb_all_q])
avg_full = np.mean([(emb_full @ eq).max() for eq in emb_all_q])

print("Average best-match similarity across all test queries:")
print(f"  No header:       {avg_none:.4f}")
print(f"  Title-only:      {avg_title:.4f}  ({(avg_title - avg_none) / avg_none * 100:+.1f}%)")
print(f"  Full hierarchy:  {avg_full:.4f}  ({(avg_full - avg_none) / avg_none * 100:+.1f}%)")
print()
print("Takeaway: Each level of context adds measurable retrieval improvement.")

---

# Part 6 — Complete RAG Pipeline with Contextual Chunk Headers

Now let's put it all together: enrich chunks with headers, build a FAISS index,
retrieve relevant context, and generate answers with the LLM.

### Pipeline Architecture

```
Document  →  Parse Structure  →  Chunk with Hierarchy  →  Prepend Headers
                                                                │
Query  →  Embed  →  FAISS Search  →  Top-k Enriched Chunks  →  LLM  →  Answer
```

In [ ]:
import faiss

# Build FAISS index from enriched embeddings
dimension = emb_enriched.shape[1]
index = faiss.IndexFlatIP(dimension)  # Inner Product = cosine similarity for normalized vectors
index.add(emb_enriched.astype(np.float32))

print(f"✅ FAISS index built: {index.ntotal} vectors, dimension={dimension}")
print(f"   Using enriched (header-prepended) embeddings.")

In [ ]:
def retrieve(query, top_k=3):
    """Retrieve top-k enriched chunks for a query using FAISS."""
    q_emb = embed_texts([query]).astype(np.float32)
    scores, indices = index.search(q_emb, top_k)
    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0])):
        results.append({
            "rank": rank + 1,
            "score": float(score),
            "enriched_text": enriched_chunks[idx]["enriched"],
            "original_text": enriched_chunks[idx]["original"],
            "hierarchy": enriched_chunks[idx]["hierarchy"],
        })
    return results

# Test retrieval
results = retrieve("What was Acme Corp revenue growth in the APAC region?")
print(f"Query: 'What was Acme Corp revenue growth in the APAC region?'\n")
for r in results:
    print(f"  #{r['rank']} [score={r['score']:.4f}] {' > '.join(r['hierarchy'])}")
    print(f"     \"{r['original_text'][:100]}...\"")
    print()

In [ ]:
def rag_query(question, top_k=3):
    """Full RAG pipeline: retrieve context with CCH, then generate answer."""
    results = retrieve(question, top_k=top_k)

    # Build context from enriched chunks (includes headers)
    context_parts = []
    for r in results:
        context_parts.append(f"[Source: {' > '.join(r['hierarchy'])}]\n{r['enriched_text']}")
    context = "\n\n---\n\n".join(context_parts)

    messages = [
        {"role": "system", "content": (
            "You are a helpful analyst. Answer the user's question based ONLY on the "
            "provided context. If the context doesn't contain enough information, say so. "
            "Cite the specific section when possible."
        )},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]

    answer = generate(messages, max_new_tokens=300, temperature=0.3)
    return answer, results

print("✅ RAG pipeline ready. Let's test it!")

### RAG Query 1: Revenue by Region

This query requires finding the specific *Regional Breakdown* section. Without headers,
chunks about headcount or R&D spending might also match "growth" and "region."

In [ ]:
question = "What was Acme Corp's revenue in the APAC region, and how fast did it grow?"
answer, sources = rag_query(question)

print(f"Q: {question}\n")
print(f"A: {answer}\n")
print("Retrieved sections:")
for s in sources:
    print(f"  #{s['rank']} [{s['score']:.4f}] {' > '.join(s['hierarchy'])}")

### RAG Query 2: Cybersecurity Spending

This is a niche query that could easily retrieve chunks about "financial investment"
or "R&D spending" without proper section context.

In [ ]:
question = "How much does Acme Corp invest in cybersecurity annually?"
answer, sources = rag_query(question)

print(f"Q: {question}\n")
print(f"A: {answer}\n")
print("Retrieved sections:")
for s in sources:
    print(f"  #{s['rank']} [{s['score']:.4f}] {' > '.join(s['hierarchy'])}")

### RAG Query 3: Cross-Section Question

This query spans multiple sections — the LLM needs to synthesize from "People and Culture"
and potentially "Company Overview."

In [ ]:
question = "What are Acme Corp's diversity and inclusion metrics, and what programs support them?"
answer, sources = rag_query(question)

print(f"Q: {question}\n")
print(f"A: {answer}\n")
print("Retrieved sections:")
for s in sources:
    print(f"  #{s['rank']} [{s['score']:.4f}] {' > '.join(s['hierarchy'])}")

## Side-by-Side: RAG With vs. Without Headers

Let's build a *second* FAISS index using bare (no-header) chunks and compare the
answers from both pipelines on the same questions.

In [ ]:
# Build a bare (no-header) FAISS index
index_bare = faiss.IndexFlatIP(dimension)
index_bare.add(emb_original.astype(np.float32))

def retrieve_bare(query, top_k=3):
    q_emb = embed_texts([query]).astype(np.float32)
    scores, indices = index_bare.search(q_emb, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": float(score),
            "text": enriched_chunks[idx]["original"],
            "hierarchy": enriched_chunks[idx]["hierarchy"],
        })
    return results

def rag_query_bare(question, top_k=3):
    results = retrieve_bare(question, top_k)
    context = "\n\n---\n\n".join([r["text"] for r in results])
    messages = [
        {"role": "system", "content": (
            "You are a helpful analyst. Answer the user's question based ONLY on the "
            "provided context. If the context doesn't contain enough information, say so."
        )},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    answer = generate(messages, max_new_tokens=300, temperature=0.3)
    return answer, results

# Compare on a tricky query
question = "What regulatory risks does Acme Corp face regarding AI?"

ans_bare, src_bare = rag_query_bare(question)
ans_cch, src_cch = rag_query(question)

print(f"Q: {question}\n")
print("=" * 70)
print("ANSWER WITHOUT HEADERS:")
print("=" * 70)
print(ans_bare)
print()
print("Retrieved (bare):")
for s in src_bare:
    print(f"  [{s['score']:.4f}] {' > '.join(s['hierarchy'])} — \"{s['text'][:60]}...\"")

print()
print("=" * 70)
print("ANSWER WITH CONTEXTUAL HEADERS:")
print("=" * 70)
print(ans_cch)
print()
print("Retrieved (CCH):")
for s in src_cch:
    print(f"  [{s['score']:.4f}] {' > '.join(s['hierarchy'])} — \"{s['original_text'][:60]}...\"")

print()
print("Notice how headers help retrieve chunks from the correct Risk Factors section.")

## Batch Retrieval Evaluation

Let's systematically compare retrieval accuracy across multiple queries. We'll check
whether the correct section appears in the top-k for each query.

In [ ]:
# Queries with expected chapter/section keywords
eval_set = [
    ("What is Acme Corp's mission?",             "Company Overview"),
    ("How much revenue came from subscriptions?", "Revenue Analysis"),
    ("APAC region growth rate",                   "Regional Breakdown"),
    ("Operating margin improvement",              "Profitability"),
    ("AI Platform v3.0 features",                 "AI Platform"),
    ("How many patents were filed?",              "Research and Development"),
    ("Security certifications achieved",          "Security and Compliance"),
    ("Voluntary attrition rate",                  "Workforce Growth"),
    ("Women in workforce percentage",             "Diversity and Inclusion"),
    ("EU AI Act compliance",                      "Regulatory Environment"),
]

def eval_retrieval(index_to_use, embeddings, top_k=3):
    hits = 0
    for query, expected_keyword in eval_set:
        q_emb = embed_texts([query]).astype(np.float32)
        _, indices = index_to_use.search(q_emb, top_k)
        for idx in indices[0]:
            hierarchy_str = " > ".join(enriched_chunks[idx]["hierarchy"])
            if expected_keyword.lower() in hierarchy_str.lower():
                hits += 1
                break
    return hits / len(eval_set)

acc_bare = eval_retrieval(index_bare, emb_original)
acc_cch  = eval_retrieval(index, emb_enriched)

print(f"Retrieval Accuracy (correct section in top-3):")
print(f"  Without headers: {acc_bare:.0%} ({int(acc_bare * len(eval_set))}/{len(eval_set)})")
print(f"  With CCH:        {acc_cch:.0%} ({int(acc_cch * len(eval_set))}/{len(eval_set)})")
print()
if acc_cch > acc_bare:
    print(f"CCH improves retrieval accuracy by {(acc_cch - acc_bare):.0%} absolute.")
elif acc_cch == acc_bare:
    print("Both achieve the same accuracy — CCH still helps with ranking and disambiguation.")
else:
    print("Unexpected: bare outperformed CCH. Check the document and queries.")

In [ ]:
# --- Per-query retrieval breakdown ---
print(f"{'Query':<43} {'Expected Section':<25} {'Bare':>5} {'CCH':>5}")
print("-" * 105)

for query, expected_keyword in eval_set:
    q_emb = embed_texts([query]).astype(np.float32)

    _, idx_bare_res = index_bare.search(q_emb, 3)
    _, idx_cch_res  = index.search(q_emb, 3)

    bare_hit = any(expected_keyword.lower() in " > ".join(enriched_chunks[i]["hierarchy"]).lower()
                   for i in idx_bare_res[0])
    cch_hit  = any(expected_keyword.lower() in " > ".join(enriched_chunks[i]["hierarchy"]).lower()
                   for i in idx_cch_res[0])

    print(f"{query:<43} {expected_keyword:<25} {'  ✅' if bare_hit else '  ❌':>5} {'  ✅' if cch_hit else '  ❌':>5}")

---

# Part 7 — Synthesis and Best Practices

## When Do Contextual Chunk Headers Help Most?

| Scenario | Impact | Why |
|----------|--------|-----|
| **Large multi-section documents** | 🟢 High | Chunks from different sections are easily confused without headers |
| **Multi-document corpora** | 🟢 High | Headers distinguish which *document* a chunk came from |
| **Ambiguous/pronoun-heavy text** | 🟢 High | "This increased by 15%" becomes "Revenue increased by 15%" |
| **Short, self-contained documents** | 🟡 Low | Chunks already contain enough context |
| **Code or structured data** | 🟡 Low | Structure is already embedded in the syntax |

## Overhead Considerations

**Token cost**: Headers add ~20-40 tokens per chunk. For a document with 100 chunks, that's
~2,000-4,000 extra tokens to embed. At typical embedding model speeds, this adds negligible
time.

**Storage**: Each enriched chunk is longer, so the FAISS index doesn't change (same vector
dimension), but storing the enriched text for display/generation uses more memory.

**When to use full hierarchy vs. title-only**:
- **Title-only** is a great default — it's cheap and handles multi-document disambiguation.
- **Full hierarchy** is worth it for large, deeply-structured documents (reports, manuals, legal docs).

## Production Patterns

1. **Parse at ingestion time**: Extract headings when you first process the document. Store the
   hierarchy in metadata alongside each chunk.
2. **Enrich at embedding time**: Prepend headers just before encoding. This keeps raw chunks
   clean for other uses.
3. **Include headers in LLM context**: When presenting retrieved chunks to the LLM, include
   the headers so the model knows the source context.
4. **Combine with other techniques**: CCH stacks well with reranking, HyDE, and fusion
   retrieval — each targets a different aspect of the retrieval problem.

In [ ]:
# --- Measure token overhead of headers ---
import statistics

original_lengths = [len(c["original"]) for c in enriched_chunks]
header_lengths   = [len(c["header"]) for c in enriched_chunks]
enriched_lengths = [len(c["enriched"]) for c in enriched_chunks]

overhead_pct = [(h / o) * 100 for h, o in zip(header_lengths, original_lengths)]

print("Character-level overhead analysis:")
print(f"  Avg original chunk length:  {statistics.mean(original_lengths):>6.0f} chars")
print(f"  Avg header length:          {statistics.mean(header_lengths):>6.0f} chars")
print(f"  Avg enriched chunk length:  {statistics.mean(enriched_lengths):>6.0f} chars")
print(f"  Avg overhead:               {statistics.mean(overhead_pct):>6.1f}%")
print(f"  Max overhead:               {max(overhead_pct):>6.1f}%")
print()
print("This modest overhead is well worth the retrieval quality improvement.")

## Final Demo: Putting It All Together

One last end-to-end RAG query to showcase the complete pipeline.

In [ ]:
question = "Summarize Acme Corp's financial performance in 2024, including revenue, profitability, and regional highlights."
answer, sources = rag_query(question, top_k=5)

print(f"Q: {question}\n")
print(f"A: {answer}\n")
print("=" * 70)
print("Retrieved sources (with contextual headers):")
print("=" * 70)
for s in sources:
    print(f"  #{s['rank']} [score={s['score']:.4f}] {' > '.join(s['hierarchy'])}")

---

# Conclusion

**Contextual Chunk Headers** is one of the simplest and most effective techniques for
improving RAG retrieval quality. In this notebook we:

1. **Demonstrated the lost context problem** — chunks that become ambiguous after splitting.
2. **Built a hierarchy parser** that extracts document structure from headings.
3. **Implemented CCH from scratch** — mapping chunks to their hierarchy and prepending headers.
4. **Measured the impact** — headers improve best-match similarity and increase the "signal gap"
   between relevant and irrelevant chunks.
5. **Compared graded headers** — even title-only headers help; full hierarchy helps most.
6. **Built a complete RAG pipeline** — header-enriched FAISS index with LLM generation.
7. **Evaluated systematically** — CCH improves section-level retrieval accuracy.

### Key Takeaways

- **CCH is low-cost, high-impact**: a few extra tokens per chunk yield large retrieval gains.
- **Headers disambiguate**: they tell the embedding model *what topic* the chunk belongs to.
- **Works with any embedding model and vector store**: no special infrastructure needed.
- **Stacks with other RAG techniques**: combine with reranking, HyDE, query expansion, etc.
- **Most valuable for** structured, multi-section documents in multi-document corpora.